# Who dominated the Premier League, 2000-2025?

Ask ten fans who the best Premier League player of the last twenty-five years
was and you will get ten answers, delivered with total confidence and almost no
shared definition of the question.

That is the real problem. "Best" is not one question, it is at least five:
best at their peak, best over a career, best per minute, best when it mattered,
best relative to the team around them. Those have *different answers*, and an
argument that does not say which one it means cannot be settled by evidence.

This notebook picks one definition and follows it all the way to a conclusion:

> **A player's rating is their best five consecutive seasons of goals and
> assists per 90 minutes, measured against the players they actually played
> against, discounted when the sample is small.**

That is a choice, not a discovery. By the end you will be able to see exactly
which parts of the answer depend on it.

In [ ]:
import warnings

import matplotlib
import pandas as pd

from gambeta import tifo

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.figsize"] = (8, 5)

SAMPLE = "../data/sample"
seasons = pd.read_parquet(f"{SAMPLE}/player_season_scored.parquet")
ratings = pd.read_parquet(f"{SAMPLE}/peak5.parquet")

first = tifo.season_label(seasons["season"].min())
last = tifo.season_label(seasons["season"].max())

print(f"{len(seasons):,} player-seasons")
print(f"{seasons['season'].nunique()} seasons: {first} to {last}")
print(f"{seasons['player_id'].nunique():,} distinct players")

## What is in the data, and what is not

Before any analysis, an honest inventory. This is the single most skipped step
in data science and the one that causes the most wrong conclusions.

**Present:** goals, assists, minutes, appearances, cards, club, season — for
every player in every Premier League season from 2000-01 to 2024-25.

**Absent, and unavailable at any price:** expected goals, progressive passes,
shot-creating actions, defensive actions, pressures. None of it was recorded
before 2017-18. This is why the rating uses goals and assists: not because they
are the best measure of a footballer, but because they are the only measure that
exists across the whole period. A metric available for one era and not another
cannot compare them.

**A consequence worth stating loudly:** this rating measures attacking output.
It will rank a good striker above a great centre-back, every time. That is a
known bias, not a hidden one.

In [ ]:
seasons.head()

## The naive answer, and why it is wrong

Start with the obvious approach: rank by raw goals and assists per 90 minutes.

Watch what happens.

In [ ]:
naive = (
    seasons[seasons["minutes"] >= 900]
    .nlargest(10, "ga_p90")[["player", "season", "minutes", "ga_p90"]]
    .assign(season=lambda d: d["season"].map(tifo.season_label))
)
naive

Now look at *which seasons* those come from. Scoring rates are not stable
across twenty-five years — the league changes. If goals are easier to come by in
2024 than in 2004, a raw per-90 ranking is partly a list of who played recently.

In [ ]:
by_season = (
    seasons[seasons["minutes"] >= 900]
    .groupby("season")["ga_p90"]
    .agg(["mean", "std", "count"])
    .rename(columns={"mean": "league mean G+A/90", "std": "spread", "count": "players"})
)
by_season.index = [tifo.season_label(s) for s in by_season.index]
by_season.round(3)

In [ ]:
ax = by_season["league mean G+A/90"].plot(
    marker="o", color=tifo.LIGHT["accent"], linewidth=2, markersize=6
)
ax.set_title(
    "League-average goals + assists per 90, by season", loc="left", fontweight="bold", fontsize=12
)
ax.set_xlabel("Season")
ax.set_ylabel("G+A per 90")
ax.tick_params(axis="x", rotation=60)
ax.figure.tight_layout()

The baseline moves. Any ranking built on raw rates is therefore comparing
players partly on *when they happened to play*, which is not a football
achievement.

## The fix: measure against contemporaries

The standard answer is to express each player's rate as a **standard score**
within their own season — how far above or below their peers they were, in units
of the spread among those peers. A player two standard deviations above the mean
in 2004 and one two standard deviations above the mean in 2024 were equally
dominant *relative to the football being played around them*.

The method is derived properly in the [z-scores chapter](02-z-scores-across-eras.ipynb).
Here we just use it and check that it did what we wanted.

In [ ]:
check = seasons.groupby("season")["ga_p90_z"].agg(["mean", "std"]).round(6)
print("Every season now centres on zero:")
print(f"  largest absolute season mean: {check['mean'].abs().max():.2e}")
print(f"  season spreads range from {check['std'].min():.3f} to {check['std'].max():.3f}")

## The ranking

Applying the peak-five-consecutive-seasons definition to the normalized scores.

The bars are 95% bootstrap confidence intervals — a measure of how much the
answer would wobble if the same career had been played again with the same
underlying ability. **Where two players' bars overlap substantially, the data
does not separate them**, and saying one is better than the other is opinion
wearing a number.

In [ ]:
ratings.head(15).assign(
    start=lambda d: d["start_season"].map(tifo.season_label),
    end=lambda d: d["end_season"].map(tifo.season_label),
)[["player", "score", "lo", "hi", "start", "end", "seasons_used"]].round(3)

In [ ]:
fig = tifo.ranked_dots(ratings, top=20)

## Where the data refuses to decide

This is the part most rankings hide. Two players whose intervals overlap are not
ranked by the evidence — they are ranked by rounding.

Below, every pair in the top ten whose intervals overlap. For those pairs, the
honest statement is "indistinguishable", not "7th and 8th".

In [ ]:
top10 = ratings.head(10).reset_index(drop=True)
overlaps = [
    (top10.loc[i, "player"], top10.loc[j, "player"])
    for i in range(len(top10))
    for j in range(i + 1, len(top10))
    if top10.loc[i, "lo"] <= top10.loc[j, "hi"] and top10.loc[j, "lo"] <= top10.loc[i, "hi"]
]
print(f"{len(overlaps)} indistinguishable pairs in the top ten:")
for a, b in overlaps[:12]:
    print(f"  {a}  <->  {b}")

## What would change my mind

A conclusion is only worth as much as the conditions under which the author
would abandon it. Mine, specifically:

1. **Add defensive contribution.** The rating is attacking output. A version that
   valued ball recovery, duels and progressive defending would rank different
   players, and I would expect defenders and holding midfielders to move up
   sharply. The current answer is "best attacking contributor", stated as "best
   player" only because the alternative data does not exist before 2017.

2. **Weight longevity over peak.** Peak-five deliberately ignores what a player
   did outside their best window. A career-total lens would favour players with
   fifteen good seasons over those with five extraordinary ones, and that is a
   defensible preference I simply did not choose here.

3. **Include other competitions.** A Premier League–only rating cannot see the
   Champions League or international football. Players whose defining
   performances happened elsewhere are invisible to it.

4. **Adjust for team strength.** Playing in a dominant side inflates goals and
   assists. Strength-of-schedule data is ingested but not yet used by this lens;
   folding it in would compress the advantage of players at the strongest clubs.

If someone disagrees with the ranking, the productive question is not "is this
wrong" but **"which of these four would you change, and why"**. That is an
argument that can actually make progress.